

# Install Python bindings for Tesseract and pandas

!pip install pandas
!pip install pytesseract
%pip install openpyxl
%pip install opencv-contrib-python


In [1]:
import os, sys, re, posixpath
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as  pd
from urllib.parse import urljoin

In [2]:
workingDir = Path.cwd()
instructionPathConstructor = lambda _project, _s, _mp :  _mp + _s + _project + _s + 'instructions'
_pathSeperator = '/' # path seperator for tatool project path construction, use \\ for Windows System.
PROJECT_DIR = _pathSeperator.join(str(workingDir).split(_pathSeperator)[:-1]+["app", "projects", "public"]) #split path string by seperator and retrieve folderNames except current space
TATOOL_ASSETS_PROJECT_PARENT = "https://raw.githubusercontent.com/tatool/tatool-web/master/app/projects/public/"
projectInstMap = {k: [u for u in os.listdir(instructionPathConstructor(k, _pathSeperator, PROJECT_DIR)) if len(re.findall(r"\b\w{2,3}_", u)) == 0]
 for k in os.listdir(PROJECT_DIR)} 
def deconstructFile(_file, _fullPath, _projectName):
    pattern = re.compile(r"^(.*?)_(.*?)_(\d+.*?)\.htm$", re.IGNORECASE)
    fileContent = Path(_fullPath).read_text(encoding="utf-8")
    if not fileContent:
        print("Content Empty")
        return
    soup = BeautifulSoup(fileContent, "html.parser")
    textContent = soup.get_text(separator=" ", strip=True)
    match = pattern.match(_file)
    if not match:
        print(_file)
        return
    imageURL = f'=IMAGE("{urljoin(TATOOL_ASSETS_PROJECT_PARENT, posixpath.join(_projectName, 'instructions', imgPath[0].get("src")))}")' if (imgPath := soup.find_all('img')) else None
    parentName, subParentName, instIndex, content = match.group(1), match.group(2), match.group(3), textContent
    del match, textContent
    return parentName, subParentName, instIndex, content, imageURL

def retrieveDisplayText(_projectList, _projectName):
    excelReadyContent = {}
    htm_files = [htm for htm in _projectList if 'htm' in htm]
    for file in htm_files:
        keys = list(excelReadyContent.keys())
        parentName, subParentName, instIndex, content, imageUrl = deconstructFile(file, _pathSeperator.join([PROJECT_DIR, _projectName, 'instructions', file]), _projectName)
        if parentName in keys:
                if subParentName in list(excelReadyContent[parentName].keys()):
                        excelReadyContent[parentName][subParentName].append({"_index": instIndex, "content": content, "ImageUrl": imageUrl})
                else:
                    excelReadyContent[parentName][subParentName] = [{"_index": instIndex, "content": content, "ImageUrl": imageUrl}]
        else:
            excelReadyContent[parentName] = {subParentName: [{"_index": instIndex, "content": content, "ImageUrl": imageUrl}]}
    return excelReadyContent

def write2Excel(_contents, outputFile):
    with pd.ExcelWriter(outputFile, engine="openpyxl") as writer:
        for projectKey, projectContent in _contents.items():
            temp_df = pd.DataFrame( 
                [{**item, "ExecutorName": exec_name}
     for exec_name, exec_content in projectContent.items()
     for item in exec_content]
            )
            temp_df.to_excel(writer, sheet_name=projectKey, index=False)
            del  temp_df

In [3]:
write2Excel(
    retrieveDisplayText(projectInstMap['uzh-ef-battery'], 'uzh-ef-battery'), 
    "uzh-ef-battery-instructions.xlsx")